In [ ]:
!pip install accelerate

In [3]:
import os
import json
import torch
import numpy as np
import networkx as nx
from typing import List
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
HF_TOKEN = "hf_" # Change with your token

#### 1. Load Knowledge Graph from JSON folder

In [ ]:
def load_kg_from_folder(folder_path: str) -> nx.DiGraph:
    G = nx.DiGraph()
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            with open(os.path.join(folder_path, filename), 'r') as f:
                data = json.load(f)
            for record in data:
                path = record.get("p", {})
                start = path.get("start", {}).get("properties", {}).get("name")
                end = path.get("end", {}).get("properties", {}).get("name")
                rel = path.get("segments", [])[0].get("relationship", {}).get("type", "RELATED_TO")
                if start and end:
                    G.add_node(start)
                    G.add_node(end)
                    G.add_edge(start, end, relation=rel)
    return G

#### 2. Retrieve context from KG

In [ ]:
def retrieve_kg_context(entity: str, G: nx.DiGraph, depth: int = 1) -> list:
    context = set()
    if entity not in G:
        return []
    nodes = [entity]
    for _ in range(depth):
        next_nodes = []
        for node in nodes:
            neighbors = list(G.successors(node))
            for nbr in neighbors:
                rel = G.get_edge_data(node, nbr).get("relation", "RELATED_TO")
                context.add(f"{node} -[{rel}]-> {nbr}")
                next_nodes.append(nbr)
        nodes = next_nodes
    return list(context)

##### 3. Download model Llama-3.2-1B-Instruct (gated repo)

In [ ]:
generator_model_id = "meta-llama/Llama-3.2-3B-Instruct"
generator_tokenizer = AutoTokenizer.from_pretrained(
    generator_model_id,
    token=HF_TOKEN
)
generator_model = AutoModelForCausalLM.from_pretrained(
    generator_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)

#### 4. Generate answers with Llama-3.2-1B-Instruct

In [ ]:
def generate_response(query: str, context: list) -> str:
    prompt = f"[INST] Context:\n" + "\n".join(context) + f"\n\nQuestion: {query}\nAnswer: [/INST]"
    inputs = generator_tokenizer(prompt, return_tensors="pt", truncation=True).to(generator_model.device)
    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=512,
        pad_token_id=generator_tokenizer.eos_token_id
    )
    return generator_tokenizer.decode(outputs[0], skip_special_tokens=True)

#### 5. Embedding model (using roberta-base for evaluation) 

In [ ]:
embed_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
embed_model = AutoModel.from_pretrained("roberta-base")

def get_embedding(text: str) -> np.ndarray:
    inputs = embed_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

def calculate_reward(response: str, ground_truth: str) -> float:
    vec1 = get_embedding(response)
    vec2 = get_embedding(ground_truth)
    return cosine_similarity([vec1], [vec2])[0][0]

#### 6. Simple RL function (step-by-step)

In [ ]:
def rl_step(query: str, ground_truth: str, G: nx.DiGraph):
    context = retrieve_kg_context(query, G)
    response = generate_response(query, context)
    reward = calculate_reward(response, ground_truth)
    return response, reward

#### 7. Reinforcement Learning loop training

In [ ]:
def train_rl(query: str, ground_truth: str, G: nx.DiGraph, episodes: int = 5):
    best_reward = -1
    best_response = ""
    for i in range(episodes):
        response, reward = rl_step(query, ground_truth, G)
        print(f"[Episode {i+1}] Reward: {reward:.4f}")
        if reward > best_reward:
            best_reward = reward
            best_response = response
    return best_response


#### 8. Define state for later expansion 

In [ ]:
def define_state(query: str, context_chunks: List[str], rewritten_query: str = None, previous_responses: List[str] = None, previous_rewards: List[float] = None) -> dict:
    state = {
        "original_query": query,
        "current_query": rewritten_query if rewritten_query else query,
        "context": context_chunks,
        "previous_responses": previous_responses if previous_responses else [],
        "previous_rewards": previous_rewards if previous_rewards else []
    }
    return state

#### 9. Main

In [ ]:
if __name__ == "__main__":
    # Replace with the path to the directory containing your JSON file
    kg_folder = "data-kg"
    G = load_kg_from_folder(kg_folder)
    
    # Example query and ground truth
    query = "Ampicillin"
    ground_truth = "Ampicillin is used to treat respiratory and urinary tract infections."
    
    # Run RL
    best_response = train_rl(query, ground_truth, G, episodes=5)
    print(f"Best Response: {best_response}")